# Combining the arms

Two steps. The first needs a GPU for a few minutes; the second runs anywhere in seconds.

1. **`src.logits`** - score every checkpoint you hold and save the model's *raw confidence*
   for each test image, over all eight orientations of the dihedral group.
2. **`src.ensemble`** - average those scores across arms, orientations and seeds, and report
   the transfer numbers.

**Why the raw scores.** `train._predict` applies the decision threshold internally and returns
yes/no; `metrics.json` keeps only accuracies. So the confidence is thrown away at the moment
it is produced. Averaging yes/no answers cannot work - two models disagreeing is a 1-1 tie,
and the confidence that would have broken it is gone.

**Why this should work at all.** The headline reports Spearman rho = 0.00 between the
`center_crop` and `rescale` generator rankings: the two arms fail on *different* generators.
That is precisely the condition under which averaging beats either member. So this is not a
fishing trip - it is a prediction the headline makes about itself, and it can come out wrong.

**Two rules fixed before any number is seen**, so the result cannot be tuned into existence:

- The rule is the **unweighted mean of the logits**. No fitted weights, no adjusted threshold.
- **`pad` is excluded**, because it is a border detector (diagonal 0.9776 against a 0.977
  pure-border ceiling). Averaging it in would raise the number using the very contamination
  the study exists to expose. `src.ensemble` refuses it rather than trusting anyone to remember.

Nothing here writes into the repo: the `.npz` goes to Drive and you move it into git yourself.

## Setup

Set `OWNER`. Everything else follows.

In [ ]:
# Re-run this after any runtime restart.
import os, sys, json, glob
from pathlib import Path

!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU on this runtime -> Runtime > Change runtime type > T4 GPU > Save, then re-run this cell (the restart unmounts Drive)"

from google.colab import drive
drive.mount('/content/drive')

OWNER = "ido"

DRIVE = "/content/drive/MyDrive/university/deep_learning"
CACHE = f"{DRIVE}/cache"

# Weights: this account's own backup, written by 01_run_matrix cell 17.
CKPT_ROOT = f"/content/drive/MyDrive/deep_learning_results/{OWNER}/runs"

# Output. Deliberately in Drive, not in the cloned repo - Ido moves what he wants into git.
OUT_DIR = f"/content/drive/MyDrive/deep_learning_results/{OWNER}/logits"
os.makedirs(OUT_DIR, exist_ok=True)

# Metrics come from git and list all 56 runs; only this account's 28 have weights here.
RESULTS = "results/runs"

assert os.path.isdir(CACHE), f"cache not found: {CACHE}"
print("cache <-", CACHE)
print("ckpts <-", CKPT_ROOT)
print("out   ->", OUT_DIR)

## Get the code

In [ ]:
![ -d /content/deep-learning ] || git clone -q https://github.com/noa-keter/deep-learning.git /content/deep-learning
%cd /content/deep-learning
# main, not the experiment branch: this analyses the 56 REPORTED runs, and anything
# computed from the experimental architecture could not go in the results section.
!git checkout main
!git pull --ff-only
!git log --oneline -1

## Gate: are the checkpoints there?

28 is a full half of the matrix. `center_crop` and `rescale` are the pair the headline rests
on and are both yours, so cross-arm combining needs nothing from Noa.

In [ ]:
found = sorted(glob.glob(f"{CKPT_ROOT}/*/*/seed*/model.pt"))
by_arm = {}
for path in found:
    by_arm.setdefault(Path(path).parts[-4], []).append(path)

print(f"{len(found)} checkpoints under {CKPT_ROOT}")
for arm, paths in sorted(by_arm.items()):
    print(f"  {arm:<13}{len(paths):>3}")
if not found:
    print("\nNOTHING FOUND. Is OWNER right, and is Drive mounted on THIS account?")

## Step 1 - dump the scores

28 checkpoints x 4,000 images x 8 orientations, about 3.6 MB. A few minutes.

**Read the reproduction line.** The upright, unmirrored pass must reproduce each run's
recorded cell accuracies - it is the same model on the same images, so it has to. The check
allows up to 4 differing rows per run, because these scores come from fp16 autocast and a
logit sitting on the threshold can land either way on a different GPU. It raises beyond that,
and if it raises, nothing in the file is about the reported models.

In [ ]:
dump_cmd = (
    f'python -m src.logits --cache-dir "{CACHE}" --ckpt-root "{CKPT_ROOT}"'
    f' --owner {OWNER} --results-dir "{RESULTS}" --out-dir "{OUT_DIR}"'
)
!{dump_cmd}

## Step 2 - combine

No GPU needed for this; it is pure arithmetic on the file just written. You can also run it
on your laptop later against a downloaded copy, and try any combination without Colab.

**Read the `d off` column** - the change in cross-generator accuracy against the first row.
That is the number this project is about. In-domain going up while `d off` stays flat is the
outcome Gate 0 already predicted and would not be worth reporting as an improvement.

The `single-arm reproduction` block above the table must match the recorded matrix:
`center_crop` at 0.908 / 0.655 and `rescale` at 0.902 / 0.563. If it does not, stop.

In [ ]:
!python -m src.ensemble --logits "{OUT_DIR}/logits_{OWNER}.npz"

## Hand off

The `.npz` is in Drive at `.../deep_learning_results/<owner>/logits/`. It is ~3.6 MB, small
enough to commit, and it is worth keeping: every future question of the form "what if we
combined X and Y" becomes a laptop calculation instead of another Colab session.

If you commit it, `results/figures/` is the right home - it is a derived analysis artifact,
not a run.

Noa can run this identical notebook with `OWNER = "noa"` to produce `logits_noa.npz`, which
would add `random_crop` as a third view. `pad` cannot join the ensemble at all.

In [ ]:
path = f"{OUT_DIR}/logits_{OWNER}.npz"
print(f"{path}\n{os.path.getsize(path) / 1e6:.1f} MB")

import numpy as np
d = np.load(path)
runs = [k for k in d.files if k not in ("y_test", "gen_ids", "orientations")]
print(f"{len(runs)} runs, orientations {list(d['orientations'])}")
print(f"test rows {len(d['y_test'])}, shape per run {d[runs[0]].shape}")